# Load extracted triples into Neo4j

Loads two JSONLs into the same graph:

- `open_extraction.jsonl` - 131 YES-contradiction pairs (real_contradiction)
- `binary_no_extraction.jsonl` - all 448 NO-contradiction pairs (contradiction_binary)

**Schema**

| label | properties |
|---|---|
| `:Entity` | `name` (normalized lowercased) |
| `:Claim` | `claim_id`, `pair_id`, `side`, `text`, `contradiction_type`, `contradiction_label`, `embedding` (384-dim MiniLM) |
| `[:RELATION]` (edge) | `predicate`, `pair_id`, `side`, `claim_id`, `contradiction_type`, `contradiction_label` |
| `[:ASSERTS]` (edge) | (no properties) - connects `:Claim` to every `:Entity` it mentions |

Claim text lives only on `:Claim.text`; each `[:RELATION]` carries a `claim_id` pointer. `(c:Claim)-[:ASSERTS]->(e:Entity)` edges let Graph RAG walk from a claim to its entities natively.

`claim_id = f"{pair_id}_{side}"`. Positive ids are integers (`1`-`131`); negative ids are `cbno_1`-`cbno_448`. Gold contradiction pairs for retrieval eval = the 131 `contradiction_label = "YES"` pairs.

In [39]:
import json
from pathlib import Path

from langchain_neo4j import Neo4jGraph
from pydantic import BaseModel, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [40]:
class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",
    )
    neo4j_uri: str = "bolt://localhost:7687"
    neo4j_user: str = "neo4j"
    neo4j_password: SecretStr


settings = Settings()
TRIPLES_FILE = Path("data/processed/kg_triples/open_extraction.jsonl")
print(f"Neo4j URI: {settings.neo4j_uri}")
print(f"Triples file: {TRIPLES_FILE.resolve()}")

Neo4j URI: bolt://localhost:7687
Triples file: D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\experiments\data\processed\kg_triples\open_extraction.jsonl


In [41]:
graph = Neo4jGraph(
    url=settings.neo4j_uri,
    username=settings.neo4j_user,
    password=settings.neo4j_password.get_secret_value(),
)
print("Connected")
print(graph.query("CALL dbms.components() YIELD name, versions RETURN name, versions"))

Connected
[{'name': 'Neo4j Kernel', 'versions': ['5.26.24']}]


In [42]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


POSITIVE_FILE = TRIPLES_FILE
NEGATIVE_FILE = Path("data/processed/kg_triples/binary_no_extraction.jsonl")


def tag_records(path: Path, default_label: str) -> list[dict]:
    records = load_jsonl(path)
    for r in records:
        r.setdefault("contradiction_label", default_label)
    return records


pos_records = tag_records(POSITIVE_FILE, "YES")
neg_records = tag_records(NEGATIVE_FILE, "NO")
records = pos_records + neg_records
total_triples = sum(len(r["t_triples"]) + len(r["h_triples"]) for r in records)
print(f"Positive pairs (YES): {len(pos_records)}")
print(f"Negative pairs (NO):  {len(neg_records)}")
print(f"Total: {len(records)} pairs, {total_triples} triples")

Positive pairs (YES): 131
Negative pairs (NO):  448
Total: 579 pairs, 2832 triples


## Embed claims

Encode each t/h sentence with `all-MiniLM-L6-v2` (384-dim, cosine). Embeddings are stored as a property on the `:Claim` node so vector search returns claim nodes directly, which then fan out to their triples via `claim_id`.

In [43]:
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
EMBEDDING_DIM = embed_model.get_embedding_dimension()

claim_texts: list[str] = []
claim_keys: list[tuple[str, str]] = []
for record in records:
    pair_id = record["id"]
    claim_texts.append(record["t_text"])
    claim_keys.append((pair_id, "t"))
    claim_texts.append(record["h_text"])
    claim_keys.append((pair_id, "h"))

embeddings = embed_model.encode(claim_texts, batch_size=32, show_progress_bar=False, convert_to_numpy=True)
claim_embeddings: dict[str, list[float]] = {f"{pair_id}_{side}": emb.tolist() for (pair_id, side), emb in zip(claim_keys, embeddings)}
print(f"Encoded {len(claim_embeddings)} claims ({EMBEDDING_DIM}-dim)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoded 1158 claims (384-dim)


## Clear existing graph (optional)

Run this cell if you want to start clean. Safe because the whole graph is derived from the JSONL file and can be rebuilt by rerunning the insert cell.

In [44]:
graph.query("MATCH (n) DETACH DELETE n")
print("Graph cleared")

Graph cleared


## Insert claims and triples

Per pair, upsert one `:Claim` per side (with precomputed embedding) and then one `[:RELATION]` per triple. `MERGE` on entity name, claim_id, and the (predicate, pair_id, side) composite key so rerunning this cell is idempotent.

In [45]:
UPSERT_CLAIM_CYPHER = """
MERGE (c:Claim {claim_id: $claim_id})
SET c.pair_id = $pair_id,
    c.side = $side,
    c.text = $text,
    c.contradiction_type = $contradiction_type,
    c.contradiction_label = $contradiction_label,
    c.embedding = $embedding
"""

INSERT_CYPHER = """
MATCH (c:Claim {claim_id: $claim_id})
MERGE (s:Entity {name: $subject})
MERGE (o:Entity {name: $object})
MERGE (s)-[r:RELATION {predicate: $predicate, pair_id: $pair_id, side: $side}]->(o)
SET r.contradiction_type = $contradiction_type,
    r.contradiction_label = $contradiction_label,
    r.claim_id = $claim_id
MERGE (c)-[:ASSERTS]->(s)
MERGE (c)-[:ASSERTS]->(o)
"""


def normalize_entity(name: str | None) -> str:
    return (name or "").strip().lower()


inserted = 0
skipped = 0
for record in tqdm(records, desc="Pairs"):
    pair_id = record["id"]
    contradiction_type = record["type"]
    contradiction_label = record.get("contradiction_label", "YES")
    for side, triples_key, text_key in (("t", "t_triples", "t_text"), ("h", "h_triples", "h_text")):
        claim_id = f"{pair_id}_{side}"
        source_text = record[text_key]
        graph.query(
            UPSERT_CLAIM_CYPHER,
            params={
                "claim_id": claim_id,
                "pair_id": pair_id,
                "side": side,
                "text": source_text,
                "contradiction_type": contradiction_type,
                "contradiction_label": contradiction_label,
                "embedding": claim_embeddings[claim_id],
            },
        )
        for triple in record[triples_key]:
            subject = normalize_entity(triple.get("subject"))
            predicate = triple.get("predicate")
            obj = normalize_entity(triple.get("object"))
            if not subject or not predicate or not obj:
                skipped += 1
                continue
            graph.query(
                INSERT_CYPHER,
                params={
                    "subject": subject,
                    "predicate": predicate,
                    "object": obj,
                    "pair_id": pair_id,
                    "side": side,
                    "claim_id": claim_id,
                    "contradiction_type": contradiction_type,
                    "contradiction_label": contradiction_label,
                },
            )
            inserted += 1

print(f"Inserted: {inserted} RELATION + 2 x {inserted} ASSERTS edges, {2 * len(records)} claims")
print(f"Skipped (missing subject/predicate/object): {skipped}")

Pairs: 100%|██████████| 579/579 [00:22<00:00, 26.23it/s]

Inserted: 2824 RELATION + 2 x 2824 ASSERTS edges, 1158 claims
Skipped (missing subject/predicate/object): 8


## Verify

In [46]:
counts = graph.query(
    """
    MATCH (n:Entity)
    WITH count(n) AS entities
    MATCH ()-[r:RELATION]->()
    RETURN entities, count(r) AS relations
    """
)
print("Counts:", counts)

print("\nTop predicates in the graph:")
for row in graph.query(
    """
    MATCH ()-[r:RELATION]->()
    RETURN r.predicate AS predicate, count(*) AS n
    ORDER BY n DESC LIMIT 10
    """
):
    print(f"  {row['predicate']}: {row['n']}")


def _truncate(text: str, width: int) -> str:
    if len(text) <= width:
        return text
    if width <= 3:
        return text[:width]
    return text[: width - 3] + "..."


def pretty_print_relations(rows: list[dict]) -> None:
    if not rows:
        print("  (no relations found)")
        return

    subject_w = min(max(len(str(row.get("subject", ""))) for row in rows), 28)
    predicate_w = min(max(len(str(row.get("predicate", ""))) for row in rows), 20)
    object_w = min(max(len(str(row.get("object", ""))) for row in rows), 28)
    pair_id_w = min(max(len(str(row.get("pair_id", ""))) for row in rows), 10)
    side_w = min(max(len(str(row.get("side", ""))) for row in rows), 4)

    for row in rows:
        subject = _truncate(str(row.get("subject", "")), subject_w)
        predicate = _truncate(str(row.get("predicate", "")), predicate_w)
        obj = _truncate(str(row.get("object", "")), object_w)
        pair_id = _truncate(str(row.get("pair_id", "")), pair_id_w)
        side = _truncate(str(row.get("side", "")), side_w)
        contradiction_type = str(row.get("type", ""))

        print(
            f"  ({subject:<{subject_w}}) -[:{predicate:<{predicate_w}}]-> ({obj:<{object_w}})  "
            f"pair_id={pair_id:<{pair_id_w}}  side={side:<{side_w}}  type={contradiction_type}"
        )


print("\nSample relations:")
sample_relations = graph.query(
    """
    MATCH (s)-[r:RELATION]->(o)
    RETURN s.name AS subject, r.predicate AS predicate, o.name AS object,
           r.pair_id AS pair_id, r.side AS side, r.contradiction_type AS type
    LIMIT 10
    """
)
pretty_print_relations(sample_relations)

Counts: [{'entities': 3423, 'relations': 2824}]

Top predicates in the graph:
  is: 82
  said: 64
  include: 30
  was: 23
  includes: 23
  killed: 22
  are: 21
  occurred_on: 19
  not_is: 19
  occurred_in: 12

Sample relations:
  (alexandros yiotopoulos) -[:is_age              ]-> (59 years old  )  pair_id=5   side=h  type=numeric
  (213 people            ) -[:were_killed         ]-> (null          )  pair_id=6   side=t  type=numeric
  (219 people            ) -[:were_killed         ]-> (219 people    )  pair_id=6   side=h  type=numeric
  (al-zarqawi            ) -[:was                 ]-> (palestinian   )  pair_id=7   side=t  type=WK
  (al-zarqawi            ) -[:was                 ]-> (jordanian     )  pair_id=7   side=h  type=WK
  (al-zarqawi            ) -[:was                 ]-> (right-handed  )  pair_id=8   side=t  type=antonym
  (al-zarqawi            ) -[:was                 ]-> (left-handed   )  pair_id=8   side=h  type=antonym
  (no one                ) -[:has_been_arrested

## Create vector index on `Claim.embedding`

Cosine similarity, 384-dim to match the MiniLM encoder. `IF NOT EXISTS` makes this safe to rerun.

In [47]:
graph.query(
    f"""
    CREATE VECTOR INDEX claim_embedding_idx IF NOT EXISTS
    FOR (c:Claim) ON (c.embedding)
    OPTIONS {{ indexConfig: {{
      `vector.dimensions`: {EMBEDDING_DIM},
      `vector.similarity_function`: 'cosine'
    }} }}
    """
)
for row in graph.query("SHOW VECTOR INDEXES YIELD name, state, labelsOrTypes, properties"):
    print(row)

{'name': 'claim_embedding_idx', 'state': 'ONLINE', 'labelsOrTypes': ['Claim'], 'properties': ['embedding']}


## Sample vector search

Encode a query sentence and retrieve top-k most similar `:Claim` nodes via the vector index. The claim's triples are reachable by joining `r.claim_id = c.claim_id` on the RELATION.

In [48]:
query_text = "How long did the balloonist spend in the sky?"
query_emb = embed_model.encode(query_text).tolist()

results = graph.query(
    """
    CALL db.index.vector.queryNodes('claim_embedding_idx', 5, $embedding)
    YIELD node, score
    RETURN node.claim_id AS claim_id, node.side AS side,
           node.contradiction_type AS type, node.text AS text, score
    """,
    params={"embedding": query_emb},
)

print(f"Query: {query_text}\n")
for r in results:
    print(f"  [{r['claim_id']}] score={r['score']:.3f}  type={r['type']}")
    print(f"    {r['text']}\n")

Query: How long did the balloonist spend in the sky?

  [59_t] score=0.789  type=numeric
    After spending about twelve hours in the sky, though, he came to the conclusion that he would have to shoot a few balloons after all; doing so caused him to descend slowly again, until the balloon's dangling cables got caught in a power line, causing a black out in a Long Beach neighbourhood for 20 minutes, but also allowing Walters to climb down to the ground again.

  [76_h] score=0.788  type=numeric
    After spending about twelve hours in the sky, though, he came to the conclusion that he would have to shoot a few balloons after all; doing so caused him to descend slowly again, until the balloon's dangling cables got caught in a power line, causing a black out in a Long Beach neighbourhood for 20 minutes, but also allowing Walters to climb down to the ground again.

  [59_h] score=0.785  type=numeric
    After spending about 45 minutes in the sky, though, he came to the conclusion that he w

## Example: same-subject, same-object, different predicate across sides

A starting point for surface-level contradiction retrieval. For each pair, find entity-node patterns where t-side and h-side disagree on the predicate. This is not yet a real contradiction detector - it just surfaces candidate conflicting triples inside the graph.

In [49]:
def pretty_print_contradictions(rows: list[dict]) -> None:
    if not rows:
        print("  (no contradictions found)")
        return
    for row in rows:
        pair_id = row.get("pair_id")
        ctype = row.get("type")
        subject = row.get("subject")
        print(f"  pair_id={pair_id}  type={ctype}")
        if "t_predicate" in row and "h_predicate" in row:
            obj = row.get("object")
            print(f"    t: ({subject}) -[:{row['t_predicate']}]-> ({obj})")
            print(f"    h: ({subject}) -[:{row['h_predicate']}]-> ({obj})")
        elif "t_object" in row and "h_object" in row:
            predicate = row.get("predicate")
            print(f"    t: ({subject}) -[:{predicate}]-> ({row['t_object']})")
            print(f"    h: ({subject}) -[:{predicate}]-> ({row['h_object']})")


candidates = graph.query(
    """
    MATCH (s:Entity)-[r1:RELATION]->(o:Entity)
    MATCH (s)-[r2:RELATION]->(o)
    WHERE r1.pair_id = r2.pair_id
      AND r1.side = 't' AND r2.side = 'h'
      AND r1.predicate <> r2.predicate
    RETURN r1.pair_id AS pair_id, r1.contradiction_type AS type,
           s.name AS subject, o.name AS object,
           r1.predicate AS t_predicate, r2.predicate AS h_predicate
    ORDER BY pair_id
    LIMIT 20
    """
)
print(f"Candidate conflicting triples (t vs h, same subject+object, different predicate): {len(candidates)}")
pretty_print_contradictions(candidates)

Candidate conflicting triples (t vs h, same subject+object, different predicate): 17
  pair_id=128  type=numeric
    t: (imperial war cabinet) -[:under]-> (winston churchill)
    h: (imperial war cabinet) -[:served_under]-> (winston churchill)
  pair_id=cbno_100  type=none
    t: (david davis) -[:becomes]-> (prime minister)
    h: (david davis) -[:not_will_become]-> (prime minister)
  pair_id=cbno_113  type=none
    t: (drug companies) -[:not_have_an_incentive_to_create]-> (a vaccine for hiv)
    h: (drug companies) -[:not_create]-> (a vaccine for hiv)
  pair_id=cbno_116  type=none
    t: (winkeler) -[:is_native_of]-> (beckemeyer)
    h: (winkeler) -[:not_reside_in]-> (beckemeyer)
  pair_id=cbno_119  type=none
    t: (protestors) -[:arrived_from]-> (grizinkals park)
    h: (protestors) -[:not_resident_in]-> (grizinkals park)
  pair_id=cbno_132  type=none
    t: (baldwin) -[:graduated_from]-> (butler university)
    h: (baldwin) -[:not_dismissed_by]-> (butler university)
  pair_id=cbno_

In [50]:
# Same subject+predicate, different object:
candidates = graph.query(
    """
    MATCH (s:Entity)-[r1:RELATION]->(o1:Entity)
    MATCH (s)-[r2:RELATION]->(o2:Entity)
    WHERE r1.pair_id = r2.pair_id
      AND r1.side = 't' AND r2.side = 'h'
      AND r1.predicate = r2.predicate
      AND o1.name <> o2.name
    RETURN r1.pair_id AS pair_id, r1.contradiction_type AS type,
           s.name AS subject, r1.predicate AS predicate,
           o1.name AS t_object, o2.name AS h_object
    ORDER BY pair_id
    LIMIT 20
    """
)
print(f"Candidate conflicting triples (t vs h, same subject+predicate, different object): {len(candidates)}")
pretty_print_contradictions(candidates)

Candidate conflicting triples (t vs h, same subject+predicate, different object): 20
  pair_id=104  type=antonym
    t: (dolan) -[:was_paid_for]-> (nine consecutive weeks)
    h: (dolan) -[:was_paid_for]-> (seven consecutive weeks of benefits)
  pair_id=114  type=WK_location
    t: (walters' lawnchair) -[:drifted_over]-> (los angeles)
    h: (walters' lawnchair) -[:drifted_over]-> (long beach)
  pair_id=114  type=WK_location
    t: (walters' lawnchair) -[:crossed]-> (the primary approach corridor of los angeles international airport)
    h: (walters' lawnchair) -[:crossed]-> (the primary approach corridor of long beach airport)
  pair_id=116  type=numeric
    t: (a military coup) -[:occurred_on]-> (20 january 1986)
    h: (a military coup) -[:occurred_on]-> (15 january 1986)
  pair_id=117  type=numeric
    t: (low german) -[:has_tenses]-> (five tenses)
    h: (low german) -[:has_tenses]-> (four tenses)
  pair_id=118  type=numeric
    t: (historians) -[:estimate]-> (about 20% of the whi